# Temporal-straightening fine-tuning: split primary, action width 64

Standalone Colab notebook. Load an **already widened v4 checkpoint**; no 10-to-64 migration is performed. Both camera encoders, fusion layers, and predictor are fine-tuned together with a fresh optimizer for a new run.

The existing one-step prediction loss (including its original target gradients) and SIGReg are unchanged. Add `STRAIGHTENING_WEIGHT * (1 - cosine(z[t+5]-z[t], z[t+10]-z[t+5]))`, averaged over moving triplets within each episode window. With window=3, four real observations provide two triplets. We reuse their existing embeddings; no extra encoder passes or recursive predictions are required.

**Adaptation, not an exact paper reproduction:** [Temporal Straightening for Latent Planning](https://arxiv.org/abs/2603.12231v3) regularizes visual features. Here we regularize the final fused camera-plus-proprioception embedding used by CEM, without changing the architecture. We retain SIGReg rather than transplanting the paper's stop-gradient objective. Near-stationary triplets are excluded; cosine calculations use float32. A zero loss with a low valid-triplet fraction is not evidence of useful straightness.

Set the source model directory, optional source epoch, additional EPOCHS, and loss weight in the training cell. Default weight 0.01 is an experimental starting point, not a validated optimum. For a matched control, use weight 0 with a different RUN_NAME. All original data loading, frame skip 5, ordered action blocks, AMP, compilation, optimizer settings, and BatchNorm calibration are retained.

EPOCHS always means **additional epochs**. RESUME=True restores the latest checkpoint and optimizer in this straightening run; keep source settings, weight, and epsilon unchanged. Losses are logged separately for train/validation, and best_model.zip minimizes validation **total including straightening**, not rollout failure rate. Evaluate with straightening/run_primary.ipynb. The width-64 model contract remains v4.


In [ ]:
%pip install -q h5py google-api-python-client


## Google Drive setup

Keep the same training dataset (`demo_rechunked.hdf5` or `demos_rechunked.hdf5`) in `MyDrive/primary_straight`. The mounting/download cell below is unchanged.

The default source is `MyDrive/primary_straight/finetune_action64/from_epoch_52/models/best_model.zip`. In the training cell, change SOURCE_MODEL_DIR to your actual width-64 run. SOURCE_EPOCH=None chooses best_model.zip explicitly; an integer chooses epoch_N_model.zip **within that width-64 run**, not epoch N of the original width-10 run.

New files go to `MyDrive/primary_straight/straightening/<RUN_NAME>/models` and `logs`. Existing source models are never overwritten. Use a new RUN_NAME for each weight experiment, or RESUME=True to continue the same run. Set the same RUN_NAME in the companion evaluator, which saves videos in that run's rollouts folder and still reads evals.hdf5 and subgoal_labels.json from primary_straight.


In [ ]:
import json
import shutil
from pathlib import Path

import google.auth
import h5py
from google.colab import auth, drive
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from tqdm.auto import tqdm

# The mount is still used for checkpoints, but the large dataset is not
# read through DriveFS because DriveFS can cache a second 37+ GiB copy.
drive.mount('/content/drive')
auth.authenticate_user()
credentials, _ = google.auth.default()
drive_api = build('drive', 'v3', credentials=credentials, cache_discovery=False)

def find_drive_child(parent_id, name, mime_type=None):
    query = f"'{parent_id}' in parents and name = '{name}' and trashed = false"
    if mime_type is not None:
        query += f" and mimeType = '{mime_type}'"
    result = drive_api.files().list(
        q=query, spaces='drive', orderBy='modifiedTime desc', pageSize=10,
        fields='files(id,name,size,mimeType,modifiedTime)',
    ).execute()
    files = result.get('files', [])
    return files[0] if files else None

root_id = drive_api.files().get(fileId='root', fields='id').execute()['id']
primary_folder = find_drive_child(
    root_id, 'primary_straight', 'application/vnd.google-apps.folder'
)
if primary_folder is None:
    raise FileNotFoundError('Could not find MyDrive/primary_straight through the Drive API')

drive_file = None
candidate_groups = (
    ('demo_rechunked.hdf5', 'demos_rechunked.hdf5'),
    ('demo.hdf5', 'demos.hdf5'),
)
for candidate_names in candidate_groups:
    matches = [find_drive_child(primary_folder['id'], name) for name in candidate_names]
    matches = [match for match in matches if match is not None]
    if matches:
        drive_file = max(matches, key=lambda match: match['modifiedTime'])
        break
if drive_file is None:
    raise FileNotFoundError(
        'Could not find a rechunked or standard demo HDF5 in MyDrive/primary_straight'
    )
source_size = int(drive_file['size'])

# Stage exactly one HDF5 on local fast storage for training reads.
local_dir = Path('/content/data/robot/mimicgen')
local_dir.mkdir(parents=True, exist_ok=True)
local_h5 = local_dir / 'demo.hdf5'
partial_h5 = local_dir / 'demo.hdf5.partial'

# Remove only known duplicate / interrupted local copies. Never touch Drive.
for duplicate in (local_dir / 'demo_rechunked.hdf5', local_dir / 'demos_rechunked.hdf5', local_dir / 'demos.hdf5', partial_h5):
    if duplicate.exists():
        duplicate.unlink()
        print(f'removed duplicate local file: {duplicate}')

if local_h5.exists() and local_h5.stat().st_size == source_size:
    print(f'reusing complete local copy: {local_h5}')
else:
    if local_h5.exists():
        local_h5.unlink()
        print(f'removed incomplete or outdated local copy: {local_h5}')

    free_bytes = shutil.disk_usage('/content').free
    required_bytes = source_size + 5 * 1024**3
    if free_bytes < required_bytes:
        raise RuntimeError(
            f'Not enough local disk space: {free_bytes / 1024**3:.1f} GiB free, '
            f'but at least {required_bytes / 1024**3:.1f} GiB is required. '
            'Use Runtime > Disconnect and delete runtime, then rerun.'
        )

    print(
        f"streaming MyDrive/primary_straight/{drive_file['name']} → {local_h5} "
        f'({source_size / 1024**3:.2f} GiB; no DriveFS dataset cache)'
    )
    request = drive_api.files().get_media(fileId=drive_file['id'])
    with partial_h5.open('wb') as destination:
        downloader = MediaIoBaseDownload(
            destination, request, chunksize=64 * 1024**2
        )
        done = False
        with tqdm(
            total=source_size, unit='B', unit_scale=True, unit_divisor=1024,
            desc='dataset download', dynamic_ncols=True,
        ) as progress:
            while not done:
                _, done = downloader.next_chunk(num_retries=5)
                progress.update(destination.tell() - progress.n)
    if partial_h5.stat().st_size != source_size:
        partial_h5.unlink(missing_ok=True)
        raise IOError('Local dataset copy has the wrong size; partial copy removed')
    partial_h5.replace(local_h5)

# Fail here, before training, if the staged file is not the expected dataset.
with h5py.File(local_h5, 'r') as file:
    if 'data' not in file or 'mask/train' not in file or 'mask/valid' not in file:
        raise ValueError(f'Invalid MimicGen HDF5 structure: {local_h5}')
    env_args = file['data'].attrs.get('env_args')
    if isinstance(env_args, bytes):
        env_args = env_args.decode('utf-8')
    env_name = json.loads(env_args).get('env_name') if env_args else None
    if env_name != 'Lift_D0':
        raise ValueError(f'Expected Lift_D0 training data, but found {env_name!r} in {local_h5}')
    train_episodes = len(file['mask/train'])
    valid_episodes = len(file['mask/valid'])

data_dir = Path('/content/data')
free_after = shutil.disk_usage('/content').free / 1024**3
print(f'dataset ready: {local_h5} ({local_h5.stat().st_size / 1024**3:.2f} GiB)')
print(f'environment: {env_name}; episodes: {train_episodes} train / {valid_episodes} validation; local disk free: {free_after:.1f} GiB')


In [ ]:
%%writefile /content/primary_dataset_runtime.py
import re
from bisect import bisect_right
from pathlib import Path

import h5py
import numpy as np
import torch
from torch.utils.data import Dataset

"""
PyTorch dataset over MimicGen Stack demos (robomimic-style HDF5).

Each sample contains a strided observation sequence and one ordered action block for
each transition:

  observations: [o_t, o_{t+s}, ..., o_{t + window*s}]
  actions:      [[a_t, ..., a_{t+s-1}], ...]

Here ``s`` is ``frame_skip``. Every primitive command is retained in temporal
order; no averaging or accumulation is performed. Proprio
state is eef_pos (3) + eef_quat (4) + gripper_qpos (2) = 9D. Images are
returned as uint8 CHW tensors and ImageNet-normalized on the training device.

Train vs val is selected via the HDF5 mask group (`mask/train`, `mask/valid`).
Both splits use state statistics computed only from the training demonstrations.
"""


def _demo_number(name: str) -> int:
	return int(re.search(r"(\d+)", name).group(1))


def _decode_demo_name(raw) -> str:
	if isinstance(raw, bytes):
		return raw.decode("utf-8")
	return str(raw)


def build_robot_state(pos, quat, grip):
	"""Concatenate normalized robosuite proprioception tensors."""
	quat = quat / quat.norm(dim=-1, keepdim=True).clamp_min(1e-8)
	# Preserve robosuite's temporally continuous quaternion sign. Forcing a
	# fixed hemisphere creates artificial jumps when a trajectory crosses w=0.
	return torch.cat((pos, quat, grip), dim=-1)


def normalize_robot_state(state, state_mean, state_std):
	return (state - state_mean) / state_std


def normalize_robot_images(images):
	"""Apply the ImageNet normalization used by the released LeWM encoder."""
	if images.shape[-3] != 3:
		raise ValueError(f"expected RGB images, got shape {tuple(images.shape)}")
	shape = [1] * images.ndim
	shape[-3] = 3
	mean = images.new_tensor((0.485, 0.456, 0.406)).reshape(shape)
	std = images.new_tensor((0.229, 0.224, 0.225)).reshape(shape)
	return (images - mean) / std


def compute_train_state_stats(h5_path: Path) -> tuple[torch.Tensor, torch.Tensor]:
	"""Compute 9-D proprioception statistics from training demos only."""
	count = 0
	total = torch.zeros(9, dtype=torch.float64)
	total_sq = torch.zeros(9, dtype=torch.float64)

	with h5py.File(h5_path, "r") as f:
		episodes = [_decode_demo_name(x) for x in f["mask/train"][:]]
		for ep_name in episodes:
			obs = f[f"data/{ep_name}/obs"]
			state = build_robot_state(
				torch.from_numpy(obs["robot0_eef_pos"][:]).to(torch.float64),
				torch.from_numpy(obs["robot0_eef_quat"][:]).to(torch.float64),
				torch.from_numpy(obs["robot0_gripper_qpos"][:]).to(torch.float64),
			)
			count += state.shape[0]
			total += state.sum(dim=0)
			total_sq += state.square().sum(dim=0)

	if count == 0:
		raise ValueError(f"No training observations found in {h5_path}")

	mean = total / count
	variance = (total_sq / count) - mean.square()
	std = variance.clamp_min(1e-12).sqrt().clamp_min(1e-6)
	return mean.float(), std.float()


def compute_train_action_stats(h5_path: Path):
	"""Compute primitive-action statistics and robust bounds from train demos."""
	action_arrays = []
	with h5py.File(h5_path, "r") as f:
		episodes = [_decode_demo_name(x) for x in f["mask/train"][:]]
		for ep_name in episodes:
			actions = np.asarray(f[f"data/{ep_name}/actions"][:], dtype=np.float32)
			if actions.ndim != 2:
				raise ValueError(f"{ep_name} actions must be rank 2, got {actions.shape}")
			action_arrays.append(actions)

	if not action_arrays:
		raise ValueError(f"No training actions found in {h5_path}")
	actions = np.concatenate(action_arrays, axis=0)
	mean = torch.from_numpy(actions.mean(axis=0, dtype=np.float64).astype(np.float32))
	std = torch.from_numpy(actions.std(axis=0, dtype=np.float64).astype(np.float32))
	std = std.clamp_min(1e-6)
	low = torch.from_numpy(np.quantile(actions, 0.005, axis=0).astype(np.float32))
	high = torch.from_numpy(np.quantile(actions, 0.995, axis=0).astype(np.float32))
	# Zero is a useful no-motion proposal and remains inside robosuite's bounds.
	low = torch.minimum(low, torch.zeros_like(low))
	high = torch.maximum(high, torch.zeros_like(high))
	degenerate = (high - low) < 1e-6
	low = torch.where(degenerate, mean - std, low)
	high = torch.where(degenerate, mean + std, high)
	return mean, std, low, high


class RobotPrimaryDataset(Dataset):
	def __init__(
		self,
		data_dir: Path,
		frame_skip: int,
		window: int,
		val: bool = False,
		state_mean: torch.Tensor | None = None,
		state_std: torch.Tensor | None = None,
		action_mean: torch.Tensor | None = None,
		action_std: torch.Tensor | None = None,
	):
		super().__init__()
		if frame_skip < 1:
			raise ValueError(f"frame_skip must be positive, got {frame_skip}")
		if window < 1:
			raise ValueError(f"window must be positive, got {window}")

		self.frame_skip = frame_skip
		self.window = window
		self.h5_path = Path(data_dir) / "robot" / "mimicgen" / "demo.hdf5"
		if not self.h5_path.exists():
			raise FileNotFoundError(f"Dataset not found: {self.h5_path}")

		if state_mean is None or state_std is None:
			state_mean, state_std = compute_train_state_stats(self.h5_path)
		self.state_mean = torch.as_tensor(state_mean, dtype=torch.float32)
		self.state_std = torch.as_tensor(state_std, dtype=torch.float32)
		if self.state_mean.shape != (9,) or self.state_std.shape != (9,):
			raise ValueError("state_mean and state_std must both have shape (9,)")
		if action_mean is None or action_std is None:
			action_mean, action_std, _, _ = compute_train_action_stats(self.h5_path)
		self.action_mean = torch.as_tensor(action_mean, dtype=torch.float32)
		self.action_std = torch.as_tensor(action_std, dtype=torch.float32)
		if self.action_mean.ndim != 1 or self.action_std.shape != self.action_mean.shape:
			raise ValueError("action_mean and action_std must be matching vectors")

		mask_key = "valid" if val else "train"
		self.episodes = []
		self.cumulative_windows = []
		total_windows = 0
		with h5py.File(self.h5_path, "r") as f:
			episodes = [_decode_demo_name(x) for x in f["mask"][mask_key][:]]
			episodes = sorted(episodes, key=_demo_number)

			for ep_name in episodes:
				ep_len = int(
					f[f"data/{ep_name}"].attrs.get(
						"num_samples",
						f[f"data/{ep_name}/actions"].shape[0],
					)
				)
				num_windows = max(0, ep_len - window * frame_skip)
				if num_windows:
					total_windows += num_windows
					self.episodes.append(ep_name)
					self.cumulative_windows.append(total_windows)

		# Keep one cumulative count per episode instead of one Python tuple and
		# frame list per window. This stays small even for millions of windows.
		self.total_len = total_windows
		self._h5 = None

	def __len__(self):
		return self.total_len

	def _get_h5(self):
		# DataLoader workers each lazily open their own read-only handle.
		if self._h5 is None:
			self._h5 = h5py.File(self.h5_path, "r")
		return self._h5

	def _resolve_index(self, idx):
		if idx < 0:
			idx += self.total_len
		if idx < 0 or idx >= self.total_len:
			raise IndexError(idx)

		episode_idx = bisect_right(self.cumulative_windows, idx)
		episode_start = (
			0 if episode_idx == 0 else self.cumulative_windows[episode_idx - 1]
		)
		return self.episodes[episode_idx], idx - episode_start

	def __getstate__(self):
		state = self.__dict__.copy()
		state["_h5"] = None
		return state

	def close(self):
		if getattr(self, "_h5", None) is not None:
			self._h5.close()
			self._h5 = None

	def __del__(self):
		self.close()

	def _load_images(self, obs, frame_slice, cam_key):
		# HDF5 stepped hyperslabs are several times slower than reading the
		# already-touched temporal chunks contiguously and subsampling in memory.
		imgs = torch.from_numpy(
			obs[cam_key][frame_slice.start:frame_slice.stop][::frame_slice.step]
		)
		return imgs.permute(0, 3, 1, 2)

	def _load_state(self, obs, frame_slice):
		contiguous_slice = slice(frame_slice.start, frame_slice.stop)
		step = frame_slice.step
		state = build_robot_state(
			torch.from_numpy(obs["robot0_eef_pos"][contiguous_slice][::step]).float(),
			torch.from_numpy(obs["robot0_eef_quat"][contiguous_slice][::step]).float(),
			torch.from_numpy(obs["robot0_gripper_qpos"][contiguous_slice][::step]).float(),
		)
		return normalize_robot_state(state, self.state_mean, self.state_std)

	def __getitem__(self, idx):
		ep_name, start = self._resolve_index(idx)
		f = self._get_h5()
		episode = f["data"][ep_name]
		obs = episode["obs"]
		frame_slice = slice(
			start,
			start + self.window * self.frame_skip + 1,
			self.frame_skip,
		)

		above = self._load_images(obs, frame_slice, "agentview_image")
		wrist = self._load_images(obs, frame_slice, "robot0_eye_in_hand_image")
		state = self._load_state(obs, frame_slice)

		# Each transition keeps all frame_skip primitive OSC commands in order.
		primitive_actions = torch.from_numpy(
			episode["actions"][
				start:start + self.window * self.frame_skip
			]
		).float().reshape(self.window, self.frame_skip, -1)
		if primitive_actions.shape[-1] != self.action_mean.numel():
			raise ValueError(
				f"dataset actions have size {primitive_actions.shape[-1]}, "
				f"statistics have size {self.action_mean.numel()}"
			)
		actions = (primitive_actions - self.action_mean) / self.action_std
		return above, wrist, state, actions


In [ ]:
import importlib
import sys
from pathlib import Path

module_dir = Path('/content')
if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))
importlib.invalidate_caches()
sys.modules.pop('primary_dataset_runtime', None)
_spawn_module = importlib.import_module('primary_dataset_runtime')

RobotPrimaryDataset = _spawn_module.RobotPrimaryDataset
compute_train_state_stats = _spawn_module.compute_train_state_stats
compute_train_action_stats = _spawn_module.compute_train_action_stats
build_robot_state = _spawn_module.build_robot_state
normalize_robot_state = _spawn_module.normalize_robot_state
normalize_robot_images = _spawn_module.normalize_robot_images
print('spawn-safe dataset module ready: /content/primary_dataset_runtime.py')


In [ ]:
import torch
import torch.nn.functional as F
from torch import nn


class ProjectionMLP(nn.Module):
    """LeWM projection head used after encoder and predictor features."""

    def __init__(self, input_dim: int = 192, hidden_dim: int = 2048, output_dim: int = 192):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        return self.layers(x)


class ActionEncoder(nn.Module):
    """Encode one ordered primitive-action block for AdaLN conditioning."""

    def __init__(self, input_dim: int, smoothed_dim: int = 10, embedding_dim: int = 192):
        super().__init__()
        self.smoother = nn.Conv1d(input_dim, smoothed_dim, kernel_size=1)
        self.mlp = nn.Sequential(
            nn.Linear(smoothed_dim, 4 * embedding_dim),
            nn.SiLU(),
            nn.Linear(4 * embedding_dim, embedding_dim),
        )

    def forward(self, actions):
        batch_size, sequence_length, _ = actions.shape
        actions = self.smoother(actions.transpose(1, 2)).transpose(1, 2)
        return self.mlp(actions.reshape(batch_size * sequence_length, -1)).reshape(
            batch_size, sequence_length, -1
        )


class TransformerBlock(nn.Module):
    """Pre-norm transformer block with optional AdaLN-Zero conditioning."""

    def __init__(
        self,
        d_model: int = 192,
        nheads: int = 16,
        d_action: int | None = None,
        dropout: float = 0.1,
        causal_masking: bool = True,
        adaptive: bool = True,
        dim_head: int | None = None,
        mlp_dim: int | None = None,
    ):
        super().__init__()
        self.adaptive = adaptive
        hidden_dim = mlp_dim or 4 * d_model

        self.norm_one = nn.LayerNorm(d_model, elementwise_affine=not adaptive)
        self.norm_two = nn.LayerNorm(d_model, elementwise_affine=not adaptive)
        if adaptive:
            if d_action is None:
                raise ValueError("d_action is required for an adaptive transformer block")
            self.adaLN_modulation = nn.Sequential(
                nn.SiLU(),
                nn.Linear(d_action, 6 * d_model),
            )
            nn.init.zeros_(self.adaLN_modulation[-1].weight)
            nn.init.zeros_(self.adaLN_modulation[-1].bias)

        self.attn_block = MultiHeadAttention(
            d_model=d_model,
            nheads=nheads,
            dim_head=dim_head,
            dropout=dropout,
            causal_masking=causal_masking,
        )
        self.mlp = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, d_model),
            nn.Dropout(dropout),
        )

    @staticmethod
    def _modulate(x, shift, scale):
        return x * (1 + scale) + shift

    def forward(self, z, a=None):
        if self.adaptive:
            if a is None:
                raise ValueError("action conditioning is required for an adaptive block")
            shift_attn, scale_attn, gate_attn, shift_mlp, scale_mlp, gate_mlp = (
                self.adaLN_modulation(a).chunk(6, dim=-1)
            )
            z = z + gate_attn * self.attn_block(
                self._modulate(self.norm_one(z), shift_attn, scale_attn)
            )
            z = z + gate_mlp * self.mlp(
                self._modulate(self.norm_two(z), shift_mlp, scale_mlp)
            )
        else:
            z = z + self.attn_block(self.norm_one(z))
            z = z + self.mlp(self.norm_two(z))
        return z


class MultiHeadAttention(nn.Module):
    def __init__(
        self,
        d_model: int = 192,
        nheads: int = 16,
        dim_head: int | None = None,
        dropout: float = 0.1,
        causal_masking: bool = True,
    ):
        super().__init__()
        self.dropout = dropout
        self.d_k = dim_head or (d_model // nheads)
        self.nheads = nheads
        self.inner_dim = self.nheads * self.d_k
        self.causal_masking = causal_masking

        self.W_q = nn.Linear(d_model, self.inner_dim)
        self.W_k = nn.Linear(d_model, self.inner_dim)
        self.W_v = nn.Linear(d_model, self.inner_dim)
        self.W_o = nn.Linear(self.inner_dim, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        batch_size, sequence_length, _ = x.shape
        queries = self.W_q(x).reshape(
            batch_size, sequence_length, self.nheads, self.d_k
        ).transpose(1, 2)
        keys = self.W_k(x).reshape(
            batch_size, sequence_length, self.nheads, self.d_k
        ).transpose(1, 2)
        values = self.W_v(x).reshape(
            batch_size, sequence_length, self.nheads, self.d_k
        ).transpose(1, 2)

        attention = F.scaled_dot_product_attention(
            queries,
            keys,
            values,
            dropout_p=self.dropout if self.training else 0.0,
            is_causal=self.causal_masking,
        )
        attention = attention.transpose(1, 2).reshape(
            batch_size, sequence_length, self.inner_dim
        )
        return self.drop(self.W_o(attention))


In [ ]:
from torch import nn
import torch

"""
Thought it would be fun to define the architecture of the ViT-Tiny
from scratch so here it is using the transformers I defined as well
"""

class CustomVitTiny(nn.Module):
    def __init__(self,
                 patch : int = 14,
                 nheads : int = 3,
                 nlayers : int = 12,
                 d_model : int = 192,
                 npatches : int = 256):
        super().__init__()

        self.npatches = npatches
        self.d_model = d_model

        self.conv_flattener = nn.Conv2d(in_channels=3,
                                        out_channels=d_model,
                                        kernel_size=patch,
                                        stride=patch,
                                        padding=0)
        
        self.transformer_layers = nn.ModuleList([])
        for _ in range(nlayers):
            self.transformer_layers.append(TransformerBlock(d_model=d_model, 
                                           nheads=nheads,                                            
                                           dropout=0.0,
                                           causal_masking=False,
                                           adaptive=False))
        # learned positional embeddings
        self.position_emb = nn.Parameter(torch.empty(npatches + 1, d_model))
        # learned [cls] embedding
        self.cls_emb = nn.Parameter(torch.empty(1, 1, 1, d_model))
        self.norm = nn.LayerNorm(d_model)
        nn.init.trunc_normal_(self.position_emb, std=0.02)
        nn.init.trunc_normal_(self.cls_emb, std=0.02)

    def forward(self, x):
        # shape of x: (B, N, C, H, W)
        B, N, C, H, W = x.shape
        x = x.reshape(B*N, C, H, W)
        emb = self.conv_flattener(x)
        emb = emb.reshape(B, N, self.d_model, self.npatches).permute(0, 1, 3, 2)
        # shape of emb now: (512, 3, 256, 192)
        cls_emb = self.cls_emb.expand(B, N, 1, self.d_model)
        emb = torch.cat([cls_emb, emb], dim=2)
        emb = emb + self.position_emb
        # shape of emb now: (512, 3, 257, 192)
        emb = emb.reshape(B*N, self.npatches + 1, self.d_model) # now ready to go into the transformer

        for layer in (self.transformer_layers):
           emb = layer(emb)
        emb = self.norm(emb)

        emb = emb.reshape(B, N, self.npatches + 1, self.d_model)
        emb = emb[:, :, 0, :] # shape : (512, 3, 192), this is what the predictor expects as the input

        return emb





# TEST CODE for Shape
# dummy_conv = nn.Conv2d(in_channels=3,
#           out_channels=192,
#           kernel_size=14,
#           stride=14,
#           padding=0)

# dummy_frame = torch.ones((512, 3, 3, 224, 224))
# dummy_frame = dummy_frame.reshape(512*3, 3, 224, 224)
# emb = dummy_conv(dummy_frame)

# print(emb.shape)
# print("after reshaping")
# emb = emb.reshape((512, 3, 192, 256))
# emb = emb.permute(0, 1, 3, 2)
# print(emb.shape)
# cls = torch.ones(1, 1, 1, 192)
# cls = cls.expand(512, 3, 1, cls.shape[-1])
# emb = torch.cat([cls, emb], dim=2)
# print(emb.shape)
# emb = emb[:, :, 0, :]
# print(emb.shape)

# TEST CODE for the Architecture
# test = CustomVitTiny()
# dummy = torch.ones(2, 3, 3, 224, 224)
# dummy = test(dummy)
# print(dummy.shape)


In [ ]:
from torch import nn
import torch

"""
This is the network definition for the encoder network which in the paper
was just ViT-Tiny which is a visual transformer with the following parameters:
- patch size of 14, 12 layers, 3 attention heads, and a hidden dimension of 192
- there is a "projection" layer added to the end of this because visual transformers
  typically have layernorms at the end but this ruins the regularization used to
  prevent representation collapse
    - this projection layer is a single layer with batch normalization to undo the
      affects of the layernorm at the end
"""
# right now the image size is defaulted to 224x224

class EncoderNetwork(nn.Module):
    def __init__(self,
                 patch : int = 14,
                 nheads : int = 3,
                 nlayers : int = 12,
                 d_model : int = 192,
                 npatches : int = 256,
                 projector_hidden_dim: int = 2048,
                 project_output: bool = True): # npatches is (H/patch_size)*(W/patch_size)
        super().__init__()
        self.d_model = d_model
        
        self.vit = CustomVitTiny(patch, nheads, nlayers, d_model, npatches)
        self.projector = (
            ProjectionMLP(d_model, projector_hidden_dim, d_model)
            if project_output else nn.Identity()
        )
    
    def forward(self, x):
        # shape of input (x): (B, N, C, H, W)
        B, N, C, H, W = x.shape
        z = self.vit(x)
        #projection step
        z = z.reshape(B*N, self.d_model)
        z = self.projector(z)
        z = z.reshape(B, N, self.d_model)

        return z
    

# TEST Code
# dummy_image = torch.ones(24, 3, 3, 224, 224)
# enc = EncoderNetwork()
# print(enc(dummy_image).shape)


In [ ]:
from torch import nn
import torch


class RobotEncoder(nn.Module):
    def __init__(self,
                 patch : int = 14,
                 nheads : int = 3,
                 nlayers : int = 12,
                 d_model : int = 192,
                 d_state : int = 9,
                 d_state_emb : int = 64,
                 npatches : int = 36,
                 projector_hidden_dim: int = 2048):  # 84x84, patch 14 → 6x6 patches
        super().__init__()
        encoder_kwargs = dict(
            patch=patch,
            nheads=nheads,
            nlayers=nlayers,
            d_model=d_model,
            npatches=npatches,
            projector_hidden_dim=projector_hidden_dim,
            project_output=False,
        )
        # The architectures are identical, but no parameters are shared. This
        # lets each visual backbone specialize to its camera's representation.
        self.agent_encoder = EncoderNetwork(**encoder_kwargs)
        self.wrist_encoder = EncoderNetwork(**encoder_kwargs)

        # small projection network for the 
        self.state_encoder = nn.Sequential(
            nn.Linear(in_features=d_state, out_features=2*d_state),
            nn.GELU(),
            nn.Linear(in_features=2*d_state, out_features=d_state_emb)
        )

        self.fusion_mlp = nn.Sequential(
            nn.Linear(in_features=(d_model * 2 + d_state_emb), out_features=(d_model * 2)),
            nn.GELU(),
            nn.Linear(in_features=(d_model * 2), out_features=d_model),
        )
        self.projector = ProjectionMLP(d_model, projector_hidden_dim, d_model)

    def forward(self, above, wrist, state):
        # above : [Batch, Window, 3, 84, 84] | wrist : [Batch, Window, 3, 84, 84] | state : [Batch, Window, 9]
        above_emb = self.agent_encoder(above)
        wrist_emb = self.wrist_encoder(wrist)
        state_emb = self.state_encoder(state) # [Batch, Window, 64]
        fusion = torch.cat((above_emb, wrist_emb, state_emb), dim=-1) # [Batch, Window, 448]
        fusion = self.fusion_mlp(fusion)
        flat = fusion.reshape(-1, fusion.shape[-1])
        return self.projector(flat).reshape_as(fusion) # [Batch, Window, 192]


In [ ]:
import torch
from torch import nn





class PredictorNetwork(nn.Module):
    """Action-conditioned latent predictor matching the released LeWM design."""

    def __init__(
        self,
        d_model: int = 192,
        nheads: int = 16,
        d_action: int = 2,
        dropout: float = 0.1,
        nlayers: int = 6,
        window: int = 3,
        dim_head: int = 64,
        mlp_dim: int = 2048,
        action_smoothing_dim: int = 10,
        projector_hidden_dim: int = 2048,
    ):
        super().__init__()
        self.d_model = d_model
        self.d_action = d_action
        self.window = window
        self.action_encoder = ActionEncoder(
            input_dim=d_action,
            smoothed_dim=action_smoothing_dim,
            embedding_dim=d_model,
        )
        self.transformer_layers = nn.ModuleList(
            [
                TransformerBlock(
                    d_model=d_model,
                    nheads=nheads,
                    d_action=d_model,
                    dropout=dropout,
                    causal_masking=True,
                    adaptive=True,
                    dim_head=dim_head,
                    mlp_dim=mlp_dim,
                )
                for _ in range(nlayers)
            ]
        )
        self.position_emb = nn.Parameter(torch.empty(1, window, d_model))
        nn.init.trunc_normal_(self.position_emb, std=0.02)
        self.final_norm = nn.LayerNorm(d_model)
        self.projector = ProjectionMLP(d_model, projector_hidden_dim, d_model)

    def forward(self, z, a):
        batch_size, sequence_length, embedding_dim = z.shape
        if embedding_dim != self.d_model:
            raise ValueError(f"expected embedding size {self.d_model}, got {embedding_dim}")
        if sequence_length > self.window:
            raise ValueError(f"sequence length {sequence_length} exceeds window {self.window}")
        if a.shape[:2] != z.shape[:2] or a.shape[-1] != self.d_action:
            raise ValueError(
                f"expected actions [B, {sequence_length}, {self.d_action}], got {tuple(a.shape)}"
            )

        z = z + self.position_emb[:, :sequence_length]
        action_embedding = self.action_encoder(a)
        for layer in self.transformer_layers:
            z = layer(z, action_embedding)
        z = self.final_norm(z)
        z = self.projector(z.reshape(batch_size * sequence_length, self.d_model))
        return z.reshape(batch_size, sequence_length, self.d_model)


In [ ]:
import torch
from torch import nn
import numpy as np

"""
This is where SIGReg (Sketched Isotropic Gaussian Regularizer) is defined:

This regulizer represents a loss in terms of how well a batch of embeddings
form an isotropic gaussian and that's just a way of saying how spread apart
the embedding space is. This is a very important term as it helps to prevent
representation collapse of the latent space which is when all embeddings passed
through the encoder collapse to a trivial solution like 0.
"""

class SIGReg(nn.Module):
    def __init__(self,
                 knots : int = 17, # the number of points being used to approximate the integral
                 num_proj : int = 1024): # number of random directions the vector will be projected in (M)
        super().__init__()
        # setting up the trapezoidal approximation of the integral in epps-pulley test

        self.num_proj = num_proj
        # this is the tensor split up into points interpolated between 0 to 3 which will be integrated over
        # only need to 0 to 3 as the gaussian essentially goes to 0 after 3 so this serves as a strong approximate
        t = torch.linspace(0, 3, knots, dtype=torch.float32)
        dt = 3 / (knots - 1)

        # these are the weights based on the trapezoid approximation [f(x0) + 2f(x1) + ... 2f(xn-1) + f(xn)]
        weights = torch.full((knots,), 2 * dt, dtype=torch.float32)
        weights[[0, -1]] = dt

        # gaussian characteristic function
        window = torch.exp(-t.square() / 2.0)

        # storing relevant values in buffer as non learnable parameters so that they move across devices with the nn Module
        self.register_buffer("t", t)
        self.register_buffer("phi", window)
        self.register_buffer("weights", weights*window)

    def forward(self, Z): # Z : (N, B, D) or (N+1, B, D) if target embeddins is being included
        A = torch.randn(Z.size(-1), self.num_proj, device=Z.device) # tensor of random directions that the embeddings will be projected onto
        A = A.div_(A.norm(p=2, dim=0)) # normalizing the tensor to be unit vectors

        # computing the epps-pulley stat
        x_t = (Z @ A).unsqueeze(-1) * self.t
        err = (x_t.cos().mean(-3) - self.phi).square() + x_t.sin().mean(-3).square()
        stat = (err @ self.weights) * Z.size(-2)

        return stat.mean() # returning the mean error across each projection


In [ ]:
import contextlib

import torch


def resolve_amp_dtype(amp_dtype: str, device) -> torch.dtype | None:
	"""Resolve autocast dtype from config.

	Supported values:
	  - auto: bf16 on GPUs with native BF16 support (Ampere+), else fp16 on CUDA
	  - bf16 / bfloat16
	  - fp16 / float16
	  - off / none: disable autocast
	"""
	if isinstance(device, str):
		on_cuda = device.startswith("cuda")
	else:
		on_cuda = device.type == "cuda"

	if not on_cuda:
		return None

	key = amp_dtype.lower()
	if key == "auto":
		return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
	if key in ("bf16", "bfloat16"):
		return torch.bfloat16
	if key in ("fp16", "float16"):
		return torch.float16
	if key in ("off", "none"):
		return None
	raise ValueError(
		f"Unknown amp_dtype {amp_dtype!r}. Use auto, bf16, fp16, or off."
	)


def autocast_context(device, amp_dtype: str):
	"""Return a torch.amp.autocast context (or nullcontext on CPU / off)."""
	dtype = resolve_amp_dtype(amp_dtype, device)
	if dtype is None:
		return contextlib.nullcontext()
	return torch.amp.autocast("cuda", dtype=dtype)


def amp_dtype_label(amp_dtype: str, device) -> str:
	"""Human-readable label for logging which AMP mode is active."""
	resolved = resolve_amp_dtype(amp_dtype, device)
	if resolved is None:
		return "off"
	return "bf16" if resolved is torch.bfloat16 else "fp16"


In [ ]:
import inspect
import math
import re
import shutil
import tempfile
import zipfile
from pathlib import Path

import torch
import yaml
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from tqdm.auto import tqdm

CHECKPOINT_FORMAT_VERSION = 4
ARCHITECTURE_ID = "lewm_robot_dual_encoder_action64_v4"
ACTION_ENCODING_ID = "standardized_ordered_primitive_block_v2"
IMAGE_NORMALIZATION_ID = "imagenet"



def temporal_straightening_loss(embeddings, eps=1e-6):
    """Cosine curvature of consecutive fused latents; never mix batch episodes.

    Returns (mean curvature over moving triplets, fraction of valid triplets).
    Stationary pairs are excluded because their direction is undefined.
    Calculate in float32 even with bf16/fp16 training.
    """
    if embeddings.ndim != 3 or embeddings.shape[1] < 3:
        raise ValueError('Straightening needs [batch, time>=3, embedding] observations')
    if not math.isfinite(eps) or eps <= 0:
        raise ValueError('straightening eps must be finite and positive')
    velocity = embeddings.float()[:, 1:] - embeddings.float()[:, :-1]
    norm = torch.linalg.vector_norm(velocity, dim=-1, keepdim=True)
    unit = velocity / norm.clamp_min(eps)
    cosine = (unit[:, :-1] * unit[:, 1:]).sum(dim=-1).clamp(-1.0, 1.0)
    valid = ((norm[:, :-1, 0] > eps) & (norm[:, 1:, 0] > eps)).float()
    loss = ((1.0 - cosine) * valid).sum() / valid.sum().clamp_min(1.0)
    return loss, valid.mean()


def load_source_checkpoint(path, encoder, predictor, expected_stats):
    with zipfile.ZipFile(path, 'r') as zf, tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        loaded = {}
        for member in ('state_stats.pth', 'encoder_weights.pth', 'predictor_weights.pth'):
            zf.extract(member, tmp)
            loaded[member] = torch.load(tmp / member, map_location='cpu', weights_only=True)
    stats = loaded['state_stats.pth']
    contract = {
        'checkpoint_format_version': CHECKPOINT_FORMAT_VERSION,
        'architecture_id': ARCHITECTURE_ID,
        'action_smoothing_dim': 64,
        'action_encoding_id': ACTION_ENCODING_ID,
        'image_normalization_id': IMAGE_NORMALIZATION_ID,
        'frame_skip': 5, 'primitive_action_dim': 7, 'predictor_action_dim': 35,
    }
    for key, expected in contract.items():
        if stats.get(key) != expected:
            raise ValueError(f'Incompatible source checkpoint: {key}={stats.get(key)!r}, expected {expected!r}')
    for key, current in expected_stats.items():
        saved = torch.as_tensor(stats[key]).cpu().float()
        if saved.shape != current.shape or not torch.allclose(saved, current.cpu().float()):
            raise ValueError(f'Dataset {key} does not match the source checkpoint; use the original dataset')
    encoder.load_state_dict(loaded['encoder_weights.pth'], strict=True)
    predictor.load_state_dict(loaded['predictor_weights.pth'], strict=True)
    print(f'Loaded existing width-64 checkpoint: {path}; all weights preserved')


def load_config(config_path: Path) -> dict:
    with open(config_path) as f:
        return yaml.safe_load(f)


def _latest_epoch_checkpoint(save_dir: Path):
    checkpoints = []
    for path in save_dir.glob("epoch_*_model.zip"):
        match = re.fullmatch(r"epoch_(\d+)_model\.zip", path.name)
        if match:
            checkpoints.append((int(match.group(1)), path))
    return max(checkpoints, default=(None, None), key=lambda item: item[0])


def _load_training_state(
    path: Path, encoder, predictor, optimizer, scaler, device, expected_metadata
):
    with zipfile.ZipFile(path, "r") as zf, tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        if "training_state.pth" not in zf.namelist():
            raise ValueError(f"{path} has no resumable training state")
        zf.extract("training_state.pth", tmp)
        try:
            training_state = torch.load(
                tmp / "training_state.pth", map_location=device, weights_only=False
            )
        except TypeError:
            training_state = torch.load(tmp / "training_state.pth", map_location=device)

        for key, expected in expected_metadata.items():
            actual = training_state.get(key)
            if actual != expected:
                raise ValueError(
                    f"Checkpoint {key}={actual!r}, expected {expected!r}. "
                    "Start a new run with the corrected architecture and action contract."
                )

        zf.extract("encoder_weights.pth", tmp)
        zf.extract("predictor_weights.pth", tmp)
        encoder.load_state_dict(
            torch.load(tmp / "encoder_weights.pth", map_location=device, weights_only=True)
        )
        predictor.load_state_dict(
            torch.load(tmp / "predictor_weights.pth", map_location=device, weights_only=True)
        )

        try:
            optimizer.load_state_dict(training_state["optimizer"])
        except ValueError as exc:
            raise ValueError(
                "The checkpoint optimizer groups do not match the current optimizer. "
                "Start this run from scratch."
            ) from exc
        if training_state.get("scaler"):
            scaler.load_state_dict(training_state["scaler"])

    return training_state


def _save_checkpoint(
    zip_path: Path,
    encoder,
    predictor,
    optimizer,
    scaler,
    epoch: int,
    state_mean: torch.Tensor,
    state_std: torch.Tensor,
    action_mean: torch.Tensor,
    action_std: torch.Tensor,
    action_low: torch.Tensor,
    action_high: torch.Tensor,
    checkpoint_metadata: dict,
    results: dict,
    config: dict | None,
    best_val_loss: float,
    best_epoch: int,
):
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        encoder_path = tmp / "encoder_weights.pth"
        predictor_path = tmp / "predictor_weights.pth"
        training_path = tmp / "training_state.pth"
        state_stats_path = tmp / "state_stats.pth"

        torch.save(encoder.state_dict(), encoder_path)
        torch.save(predictor.state_dict(), predictor_path)
        torch.save(
            {
                "epoch": epoch,
                "optimizer": optimizer.state_dict(),
                "scaler": scaler.state_dict(),
                "state_mean": state_mean.cpu(),
                "state_std": state_std.cpu(),
                "action_mean": action_mean.cpu(),
                "action_std": action_std.cpu(),
                "action_low": action_low.cpu(),
                "action_high": action_high.cpu(),
                **checkpoint_metadata,
                "results": results,
                "best_val_loss": best_val_loss,
                "best_epoch": best_epoch,
            },
            training_path,
        )
        torch.save(
            {
                "state_mean": state_mean.cpu(),
                "state_std": state_std.cpu(),
                "action_mean": action_mean.cpu(),
                "action_std": action_std.cpu(),
                "action_low": action_low.cpu(),
                "action_high": action_high.cpu(),
                **checkpoint_metadata,
            },
            state_stats_path,
        )

        with zipfile.ZipFile(zip_path, "w") as zf:
            zf.write(encoder_path, "encoder_weights.pth")
            zf.write(predictor_path, "predictor_weights.pth")
            zf.write(training_path, "training_state.pth")
            zf.write(state_stats_path, "state_stats.pth")
            if config is not None:
                zf.writestr("config.yaml", yaml.safe_dump(config, sort_keys=False))


def _atomic_copy(source: Path, destination: Path):
    pending = destination.with_suffix(destination.suffix + ".tmp")
    shutil.copyfile(source, pending)
    pending.replace(destination)


def _make_grad_scaler(enabled: bool):
    try:
        return torch.amp.GradScaler("cuda", enabled=enabled)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=enabled)


def _make_optimizer(
    named_parameters,
    lr,
    betas,
    weight_decay,
    on_cuda,
    no_decay_parameters,
):
    no_decay_parameter_ids = {id(parameter) for parameter in no_decay_parameters}
    decay_parameters = []
    no_decay_parameters = []
    for name, parameter in named_parameters:
        if not parameter.requires_grad:
            continue
        # Only BatchNorm affine parameters avoid decay. LayerNorm scales and
        # ordinary biases need the original decay to prevent the ViT runaway
        # observed when every one-dimensional parameter was exempted.
        if id(parameter) in no_decay_parameter_ids:
            no_decay_parameters.append(parameter)
        else:
            decay_parameters.append(parameter)

    kwargs = {
        "lr": lr,
        "betas": betas,
        "weight_decay": 0.0,
    }
    if on_cuda and "fused" in inspect.signature(torch.optim.AdamW).parameters:
        kwargs["fused"] = True
    optimizer = torch.optim.AdamW(
        [
            {"params": decay_parameters, "weight_decay": weight_decay},
            {"params": no_decay_parameters, "weight_decay": 0.0},
        ],
        **kwargs,
    )
    print(
        f"AdamW parameter groups: {len(decay_parameters)} decayed tensors, "
        f"{len(no_decay_parameters)} BatchNorm affine tensors without decay"
    )
    return optimizer


def _batch_norm_modules(*models):
    return [
        module
        for model in models
        for module in model.modules()
        if isinstance(module, nn.modules.batchnorm._BatchNorm)
    ]


@torch.no_grad()
def _recalibrate_batch_norm(
    encoder,
    predictor,
    dataloader,
    max_batches,
    device,
    non_blocking,
    amp_dtype,
):
    batch_norms = _batch_norm_modules(encoder, predictor)
    if max_batches <= 0 or not batch_norms:
        encoder.eval()
        predictor.eval()
        return 0

    encoder.eval()
    predictor.eval()
    momenta = [module.momentum for module in batch_norms]
    for module in batch_norms:
        module.reset_running_stats()
        # Use a cumulative average during calibration so the result does not
        # depend disproportionately on the final few shuffled batches.
        module.momentum = None
        module.train()

    batches_seen = 0
    try:
        for batch in dataloader:
            if batches_seen >= max_batches:
                break
            above, wrist, state, actions = _prepare_batch(
                batch, device, non_blocking
            )
            with autocast_context(device, amp_dtype):
                embeddings = encoder(above, wrist, state)
                predictor(embeddings[:, :-1], actions)
            batches_seen += 1
    finally:
        for module, momentum in zip(batch_norms, momenta):
            module.momentum = momentum
        encoder.eval()
        predictor.eval()

    return batches_seen


def _batch_norm_diagnostics(model):
    batch_norms = _batch_norm_modules(model)
    if not batch_norms:
        return float("nan"), float("nan"), float("nan")

    scales = torch.cat(
        [module.weight.detach().float().flatten() for module in batch_norms]
    )
    running_vars = torch.cat(
        [module.running_var.detach().float().flatten() for module in batch_norms]
    )
    return (
        scales.square().mean().sqrt().item(),
        running_vars.min().item(),
        running_vars.max().item(),
    )


def _loader_options(num_workers, persistent_workers, pin_memory,
                    prefetch_factor, in_order, timeout, start_method):
    options = {
        "num_workers": num_workers,
        "persistent_workers": persistent_workers and num_workers > 0,
        "pin_memory": pin_memory,
    }
    if num_workers > 0:
        options["prefetch_factor"] = prefetch_factor
        options["timeout"] = timeout
        options["multiprocessing_context"] = start_method
    if "in_order" in inspect.signature(DataLoader).parameters:
        options["in_order"] = in_order
    return options


def _prepare_batch(batch, device, non_blocking):
    above, wrist, state, actions = batch
    # Keep images as uint8 in workers and pinned host memory. Conversion on the
    # GPU cuts host memory and transfer traffic by 4x.
    above = above.to(device, non_blocking=non_blocking).float().mul_(1.0 / 255.0)
    wrist = wrist.to(device, non_blocking=non_blocking).float().mul_(1.0 / 255.0)
    above = normalize_robot_images(above)
    wrist = normalize_robot_images(wrist)
    state = state.to(device, non_blocking=non_blocking)
    actions = actions.to(device, non_blocking=non_blocking)
    if actions.ndim != 4:
        raise ValueError(f"expected ordered action blocks [B, W, S, A], got {actions.shape}")
    actions = actions.flatten(start_dim=2)
    return above, wrist, state, actions


def train(device,
          data_dir: Path,
          log_dir: Path,
          save_dir: Path,
          frame_skip: int,
          lr: float,
          seed: int,
          lambd: float,
          epochs: int,
          batch_size: int,
          num_workers: int,
          persistent_workers: bool,
          pin_memory: bool,
          prefetch_factor: int,
          in_order: bool,
          loader_timeout: float,
          loader_start_method: str,
          compile_model: bool,
          window_size: int,
          embedding_dim: int,
          action_dim: int,
          pred_nheads: int,
          dropout: float,
          pred_nlayers: int,
          pred_dim_head: int,
          pred_mlp_dim: int,
          action_smoothing_dim: int,
          projector_hidden_dim: int,
          patch_size: int,
          enc_nheads: int,
          enc_nlayers: int,
          enc_npatches: int,
          knots: int,
          num_proj: int,
          optimizer_betas: tuple[float, float],
          optimizer_weight_decay: float,
          gradient_clip_norm: float,
          bn_calibration_batches: int,
          amp_dtype: str,
          resume: bool = False,
          config: dict | None = None,
          source_checkpoint: Path | None = None,
          straightening_weight: float = 0.01,
          straightening_eps: float = 1e-6):

    if not math.isfinite(straightening_weight) or straightening_weight < 0:
        raise ValueError('straightening_weight must be finite and nonnegative')
    if not math.isfinite(straightening_eps) or straightening_eps <= 0:
        raise ValueError('straightening_eps must be finite and positive')
    if window_size < 2:
        raise ValueError('Straightening requires at least three observed frames (window_size >= 2)')
    if action_smoothing_dim != 64:
        raise ValueError('This fine-tuning notebook requires action_smoothing_dim=64')
    if resume:
        if _latest_epoch_checkpoint(save_dir)[1] is None:
            raise FileNotFoundError(f'No fine-tuning checkpoint to resume in {save_dir}')
    elif source_checkpoint is None or not Path(source_checkpoint).is_file():
        raise FileNotFoundError(f'Source checkpoint not found: {source_checkpoint}')
    elif any(save_dir.glob('*.zip')):
        raise ValueError('Fine-tuning checkpoints already exist; use RESUME=True or a new RUN_NAME')
    torch.manual_seed(seed)
    if isinstance(device, str):
        on_cuda = device.startswith("cuda")
    else:
        on_cuda = device.type == "cuda"

    if on_cuda:
        torch.cuda.manual_seed(seed)
        torch.backends.cudnn.benchmark = True
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True
        torch.set_float32_matmul_precision("high")

    save_dir.mkdir(parents=True, exist_ok=True)

    # Calculate these once from the train mask, then use the same values for
    # both splits and save them with every checkpoint.
    h5_path = Path(data_dir) / "robot" / "mimicgen" / "demo.hdf5"
    state_mean, state_std = compute_train_state_stats(h5_path)
    action_mean, action_std, action_low, action_high = compute_train_action_stats(h5_path)
    print(f"state mean: {state_mean.tolist()}")
    print(f"state std:  {state_std.tolist()}")
    print(f"action mean: {action_mean.tolist()}")
    print(f"action std:  {action_std.tolist()}")
    print(
        f"temporal stride: frame_skip={frame_skip} "
        f"({frame_skip} simulator steps per model transition)"
    )

    train_dataset = RobotPrimaryDataset(
        data_dir=data_dir,
        frame_skip=frame_skip,
        window=window_size,
        val=False,
        state_mean=state_mean,
        state_std=state_std,
        action_mean=action_mean,
        action_std=action_std,
    )
    val_dataset = RobotPrimaryDataset(
        data_dir=data_dir,
        frame_skip=frame_skip,
        window=window_size,
        val=True,
        state_mean=state_mean,
        state_std=state_std,
        action_mean=action_mean,
        action_std=action_std,
    )

    predictor_action_dim = frame_skip * action_dim
    checkpoint_metadata = {
        "checkpoint_format_version": CHECKPOINT_FORMAT_VERSION,
        "architecture_id": ARCHITECTURE_ID,
        "action_encoding_id": ACTION_ENCODING_ID,
        "image_normalization_id": IMAGE_NORMALIZATION_ID,
        "frame_skip": frame_skip,
        "primitive_action_dim": action_dim,
        "predictor_action_dim": predictor_action_dim,
        "action_smoothing_dim": action_smoothing_dim,
        "source_epoch": (config or {}).get("source_epoch"),
        "source_checkpoint": str(source_checkpoint),
        "training_objective": "prediction_sigreg_fused_temporal_straightening_v1",
        "straightening_space": "fused_projected_embedding",
        "straightening_weight": float(straightening_weight),
        "straightening_eps": float(straightening_eps),
    }

    loader_options = _loader_options(
        num_workers,
        persistent_workers,
        pin_memory,
        prefetch_factor,
        in_order,
        loader_timeout,
        loader_start_method,
    )
    print(
        f"DataLoader: workers={num_workers}, start_method="
        f"{loader_options.get('multiprocessing_context', 'single-process')}, "
        f"prefetch={loader_options.get('prefetch_factor', 0)}, "
        f"persistent={loader_options['persistent_workers']}"
    )
    train_dataloader = DataLoader(
        dataset=train_dataset,
        batch_size=batch_size,
        shuffle=True,
        **loader_options,
    )
    val_dataloader = DataLoader(
        dataset=val_dataset,
        batch_size=batch_size,
        shuffle=True,
        **loader_options,
    )

    predictor = PredictorNetwork(
        d_model=embedding_dim,
        nheads=pred_nheads,
        d_action=predictor_action_dim,
        dropout=dropout,
        nlayers=pred_nlayers,
        window=window_size,
        dim_head=pred_dim_head,
        mlp_dim=pred_mlp_dim,
        action_smoothing_dim=action_smoothing_dim,
        projector_hidden_dim=projector_hidden_dim,
    ).to(device)
    encoder = RobotEncoder(
        patch=patch_size,
        nheads=enc_nheads,
        nlayers=enc_nlayers,
        d_model=embedding_dim,
        npatches=enc_npatches,
        projector_hidden_dim=projector_hidden_dim,
    ).to(device)

    if not resume:
        load_source_checkpoint(source_checkpoint, encoder, predictor, {
            'state_mean': state_mean, 'state_std': state_std,
            'action_mean': action_mean, 'action_std': action_std,
            'action_low': action_low, 'action_high': action_high,
        })

    gpu_name = torch.cuda.get_device_name(0) if on_cuda else "cpu"
    amp_label = amp_dtype_label(amp_dtype, device)
    print(f"networks are on device: {device} ({gpu_name})")
    print(f"AMP: {amp_label} (config amp_dtype={amp_dtype})")

    sig_reg = SIGReg(knots=knots, num_proj=num_proj).to(device)
    pred_loss_fn = nn.MSELoss()
    named_parameters = [
        (f"predictor.{name}", parameter)
        for name, parameter in predictor.named_parameters()
    ] + [
        (f"encoder.{name}", parameter)
        for name, parameter in encoder.named_parameters()
    ]
    trainable_parameters = [
        parameter for _, parameter in named_parameters if parameter.requires_grad
    ]
    batch_norm_parameters = [
        parameter
        for module in _batch_norm_modules(predictor, encoder)
        for parameter in module.parameters(recurse=False)
        if parameter.requires_grad
    ]
    optimizer = _make_optimizer(
        named_parameters,
        lr,
        optimizer_betas,
        optimizer_weight_decay,
        on_cuda,
        batch_norm_parameters,
    )
    scaler = _make_grad_scaler(enabled=on_cuda and amp_label == "fp16")

    writer = SummaryWriter(log_dir=log_dir)
    results = {
        "train_loss": [],
        "train_prediction_loss": [],
        "train_sigreg_loss": [],
        "train_straightening_loss": [],
        "train_straightening_valid_fraction": [],
        "train_grad_norm": [],
        "val_loss": [],
        "val_prediction_loss": [],
        "val_sigreg_loss": [],
        "val_straightening_loss": [],
        "val_straightening_valid_fraction": [],
        "encoder_bn_gamma_rms": [],
        "encoder_bn_running_var_min": [],
        "encoder_bn_running_var_max": [],
    }
    start_epoch = 0
    best_val_loss = float("inf")
    best_epoch = -1

    if resume:
        last_epoch, checkpoint_path = _latest_epoch_checkpoint(save_dir)
        if checkpoint_path is None:
            print(f"No epoch_*_model.zip found in {save_dir}; starting from scratch")
        else:
            training_state = _load_training_state(
                checkpoint_path,
                encoder,
                predictor,
                optimizer,
                scaler,
                device,
                checkpoint_metadata,
            )
            if not training_state:
                raise ValueError(
                    f"{checkpoint_path} contains weights only and cannot resume "
                    "optimizer state. Start a new run or use a new-format checkpoint."
                )
            saved_mean = training_state["state_mean"].cpu()
            saved_std = training_state["state_std"].cpu()
            if not torch.allclose(saved_mean, state_mean) or not torch.allclose(saved_std, state_std):
                raise ValueError("Dataset state statistics do not match the checkpoint")
            for key, current in (
                ("action_mean", action_mean),
                ("action_std", action_std),
                ("action_low", action_low),
                ("action_high", action_high),
            ):
                saved = training_state.get(key)
                if saved is None or not torch.allclose(saved.cpu(), current):
                    raise ValueError(f"Dataset {key} does not match the checkpoint")
            results = training_state.get("results", results)
            start_epoch = int(training_state.get("epoch", last_epoch)) + 1
            previous_val_losses = results.get("val_loss", [])
            best_val_loss = float(
                training_state.get(
                    "best_val_loss",
                    min(previous_val_losses, default=float("inf")),
                )
            )
            best_epoch = int(training_state.get("best_epoch", -1))
            if best_epoch < 0 and previous_val_losses:
                best_epoch = min(
                    range(len(previous_val_losses)),
                    key=previous_val_losses.__getitem__,
                )
            print(f"resumed {checkpoint_path.name}; continuing at epoch {start_epoch}")

    if compile_model and on_cuda:
        if hasattr(encoder, "compile") and hasattr(predictor, "compile"):
            encoder.compile()
            predictor.compile()
            print("torch.compile enabled for encoder and predictor")
        else:
            print("torch.compile requested but unavailable; using eager mode")

    non_blocking = on_cuda and pin_memory

    print(f"Training epochs [{start_epoch}, {start_epoch + epochs})")
    for epoch in tqdm(range(start_epoch, start_epoch + epochs)):
        print(f"Epoch: {epoch}\n------------")

        train_metrics = torch.zeros(6, device=device)

        encoder.train()
        predictor.train()
        for batch in train_dataloader:
            above, wrist, state, actions = _prepare_batch(
                batch, device, non_blocking
            )

            optimizer.zero_grad(set_to_none=True)
            with autocast_context(device, amp_dtype):
                # Encode o_t ... o_{t+window} once, then use the same shifted
                # prediction objective as the original LeWM training process.
                all_embeddings = encoder(above, wrist, state)
                pred_embeddings = predictor(all_embeddings[:, :-1], actions)
                target_embeddings = all_embeddings[:, 1:]

                pred_loss = pred_loss_fn(pred_embeddings, target_embeddings)
                reg_loss = sig_reg(all_embeddings.transpose(0, 1))
                straight_loss, straight_valid = temporal_straightening_loss(
                    all_embeddings, eps=straightening_eps
                )
                loss = pred_loss + lambd * reg_loss + straightening_weight * straight_loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(
                trainable_parameters,
                max_norm=gradient_clip_norm,
                error_if_nonfinite=True,
            )
            scaler.step(optimizer)
            scaler.update()

            train_metrics += torch.stack(
                (
                    loss.detach(),
                    pred_loss.detach(),
                    reg_loss.detach(),
                    straight_loss.detach(),
                    straight_valid.detach(),
                    grad_norm.detach(),
                )
            )

        num_train_batches = len(train_dataloader)
        train_total, train_prediction, train_sigreg, train_straight, train_valid, train_grad_norm = (
            train_metrics.div(num_train_batches).cpu().tolist()
        )

        calibration_batches_seen = _recalibrate_batch_norm(
            encoder=encoder,
            predictor=predictor,
            dataloader=train_dataloader,
            max_batches=bn_calibration_batches,
            device=device,
            non_blocking=non_blocking,
            amp_dtype=amp_dtype,
        )
        bn_gamma_rms, bn_var_min, bn_var_max = _batch_norm_diagnostics(encoder)

        val_metrics = torch.zeros(5, device=device)

        encoder.eval()
        predictor.eval()
        with torch.inference_mode():
            for batch in val_dataloader:
                above, wrist, state, actions = _prepare_batch(
                    batch, device, non_blocking
                )

                with autocast_context(device, amp_dtype):
                    all_embeddings = encoder(above, wrist, state)
                    pred_embeddings = predictor(all_embeddings[:, :-1], actions)
                    target_embeddings = all_embeddings[:, 1:]

                    pred_loss = pred_loss_fn(pred_embeddings, target_embeddings)
                    reg_loss = sig_reg(all_embeddings.transpose(0, 1))
                    straight_loss, straight_valid = temporal_straightening_loss(
                        all_embeddings, eps=straightening_eps
                    )
                    loss = pred_loss + lambd * reg_loss + straightening_weight * straight_loss

                val_metrics += torch.stack((loss, pred_loss, reg_loss, straight_loss, straight_valid))

        num_val_batches = len(val_dataloader)
        val_total, val_prediction, val_sigreg, val_straight, val_valid = (
            val_metrics.div(num_val_batches).cpu().tolist()
        )

        results["train_loss"].append(train_total)
        results["train_prediction_loss"].append(train_prediction)
        results["train_sigreg_loss"].append(train_sigreg)
        results["train_straightening_loss"].append(train_straight)
        results["train_straightening_valid_fraction"].append(train_valid)
        results.setdefault("train_grad_norm", []).append(train_grad_norm)
        results["val_loss"].append(val_total)
        results["val_prediction_loss"].append(val_prediction)
        results["val_sigreg_loss"].append(val_sigreg)
        results["val_straightening_loss"].append(val_straight)
        results["val_straightening_valid_fraction"].append(val_valid)
        results.setdefault("encoder_bn_gamma_rms", []).append(bn_gamma_rms)
        results.setdefault("encoder_bn_running_var_min", []).append(bn_var_min)
        results.setdefault("encoder_bn_running_var_max", []).append(bn_var_max)

        writer.add_scalars(
            main_tag="Loss/Total",
            tag_scalar_dict={"train": train_total, "validation": val_total},
            global_step=epoch,
        )
        writer.add_scalars(
            main_tag="Loss/Prediction",
            tag_scalar_dict={"train": train_prediction, "validation": val_prediction},
            global_step=epoch,
        )
        writer.add_scalars(
            main_tag="Loss/SIGReg",
            tag_scalar_dict={"train": train_sigreg, "validation": val_sigreg},
            global_step=epoch,
        )
        writer.add_scalars(
            main_tag="Loss/Straightening",
            tag_scalar_dict={"train": train_straight, "validation": val_straight},
            global_step=epoch,
        )
        writer.add_scalars(
            main_tag="Straightening/ValidTripletFraction",
            tag_scalar_dict={"train": train_valid, "validation": val_valid},
            global_step=epoch,
        )
        writer.add_scalar("Optimization/GradientNorm", train_grad_norm, epoch)
        writer.add_scalar("BatchNorm/EncoderGammaRMS", bn_gamma_rms, epoch)
        writer.add_scalar("BatchNorm/EncoderRunningVarMin", bn_var_min, epoch)
        writer.add_scalar("BatchNorm/EncoderRunningVarMax", bn_var_max, epoch)

        is_best = val_total < best_val_loss
        if is_best:
            best_val_loss = val_total
            best_epoch = epoch

        zip_path = save_dir / f"epoch_{epoch}_model.zip"
        _save_checkpoint(
            zip_path=zip_path,
            encoder=encoder,
            predictor=predictor,
            optimizer=optimizer,
            scaler=scaler,
            epoch=epoch,
            state_mean=state_mean,
            state_std=state_std,
            action_mean=action_mean,
            action_std=action_std,
            action_low=action_low,
            action_high=action_high,
            checkpoint_metadata=checkpoint_metadata,
            results=results,
            config=config,
            best_val_loss=best_val_loss,
            best_epoch=best_epoch,
        )
        if is_best:
            _atomic_copy(zip_path, save_dir / "best_model.zip")

        print(
            f"Train total: {train_total:.4f} "
            f"(prediction {train_prediction:.4f}, SIGReg {train_sigreg:.4f}, "
            f"straightening {train_straight:.4f}) | "
            f"Validation total: {val_total:.4f} "
            f"(prediction {val_prediction:.4f}, SIGReg {val_sigreg:.4f}, "
            f"straightening {val_straight:.4f}) | "
            f"grad norm {train_grad_norm:.4f} | "
            f"encoder BN gamma RMS {bn_gamma_rms:.4f} "
            f"(recalibrated on {calibration_batches_seen} batches)"
        )
        if is_best:
            print(f"new best validation checkpoint → {save_dir / 'best_model.zip'}")

    train_dataset.close()
    val_dataset.close()
    writer.close()
    return results




In [ ]:
import time
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/primary_straight')
SOURCE_MODEL_DIR = DRIVE_DIR / 'finetune_action64' / 'from_epoch_52' / 'models'
SOURCE_EPOCH = None        # None = best_model.zip; integer = epoch within the width-64 run.
EPOCHS = 5                 # Additional epochs, including when resuming.
RESUME = False             # True restores this straightening run's model AND optimizer.
STRAIGHTENING_WEIGHT = 0.01  # Use 0.0 with a new RUN_NAME for the matched control.
STRAIGHTENING_EPS = 1e-6   # Ignore latent displacements with norm <= this value.
RUN_NAME = 'action64_straightening_001'

if SOURCE_EPOCH is not None and (
    not isinstance(SOURCE_EPOCH, int) or isinstance(SOURCE_EPOCH, bool) or SOURCE_EPOCH < 0
):
    raise ValueError('SOURCE_EPOCH must be None or a nonnegative integer')
if not isinstance(EPOCHS, int) or isinstance(EPOCHS, bool) or EPOCHS < 1:
    raise ValueError('EPOCHS must be a positive integer')
if not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in ('.', '..'):
    raise ValueError('RUN_NAME must be a single folder name')
source_checkpoint = Path(SOURCE_MODEL_DIR) / (
    'best_model.zip' if SOURCE_EPOCH is None else f'epoch_{SOURCE_EPOCH}_model.zip'
)
RUN_DIR = DRIVE_DIR / 'straightening' / RUN_NAME
save_dir = RUN_DIR / 'models'
log_dir = RUN_DIR / 'logs'
data_dir = Path('/content/data')
if not RESUME and RUN_DIR.exists() and any(RUN_DIR.iterdir()):
    raise ValueError(f'{RUN_DIR} already contains a run. Set RESUME=True or choose a new RUN_NAME')
if not RESUME and not source_checkpoint.is_file():
    raise FileNotFoundError(source_checkpoint)
print(f'Source: {source_checkpoint}')
print(f'Straightening output: {RUN_DIR}')
print(f'Straightening weight: {STRAIGHTENING_WEIGHT}; epsilon: {STRAIGHTENING_EPS}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
start_time = time.perf_counter()
results = train(
    device=device,
    data_dir=data_dir,
    log_dir=log_dir,
    save_dir=save_dir,
    frame_skip=5,
    lr=5e-5,
    seed=42,
    lambd=0.1,
    epochs=EPOCHS,
    batch_size=128,
    num_workers=8,
    persistent_workers=False,
    pin_memory=True,
    prefetch_factor=2,
    in_order=True,
    loader_timeout=120,
    loader_start_method='spawn',
    compile_model=True,
    window_size=3,
    embedding_dim=192,
    action_dim=7,
    pred_nheads=16,
    dropout=0.1,
    pred_nlayers=6,
    pred_dim_head=64,
    pred_mlp_dim=2048,
    action_smoothing_dim=64,
    projector_hidden_dim=2048,
    patch_size=14,
    enc_nheads=3,
    enc_nlayers=12,
    enc_npatches=36,
    knots=17,
    num_proj=1024,
    optimizer_betas=(0.9, 0.999),
    optimizer_weight_decay=0.001,
    gradient_clip_norm=1.0,
    bn_calibration_batches=256,
    amp_dtype='auto',
    resume=RESUME,
    source_checkpoint=source_checkpoint,
    straightening_weight=STRAIGHTENING_WEIGHT,
    straightening_eps=STRAIGHTENING_EPS,
    config={'source_epoch': SOURCE_EPOCH, 'source_checkpoint': str(source_checkpoint),
            'action_smoothing_dim': 64, 'learning_rate': 5e-5, 'run_name': RUN_NAME,
            'straightening_weight': STRAIGHTENING_WEIGHT, 'straightening_eps': STRAIGHTENING_EPS,
            'straightening_space': 'fused_projected_embedding'},
)
print(f'training time: {(time.perf_counter() - start_time):.2f}s')
print(f'checkpoints saved to: {save_dir}')
print(f'logs saved to: {log_dir}')


In [ ]:
%load_ext tensorboard
%tensorboard --logdir $log_dir
